# 25 — AssayGroups

Group multiple sensor channels into a named `AssayGroup` and reuse that group across studies and comparisons.

**Dataset**: first and second studies in any ISA-PHM dataset  
**API**: `AssayGroup` · `study.assay_group()` · `study.lifecycle_features(group=...)` · `group.compare_with()`
Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper, AssayGroup
from isa_phm.plotter import ISAPlotter
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show

output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    cache_maxsize=20,
)


## Dataset overview


In [ ]:
study = wrapper.study(1)  # first study (1-based index)

print(f"Study  : {study.title}")
print(f"Sensors: {[a.assay_id for a in study.list_assays()]}")


## 1. Define an AssayGroup

`AssayGroup` is a **study-agnostic** descriptor — it holds a list of assay IDs (or 1-based integer indices) and a name.  
Define it once and bind it to any study with `study.assay_group(group)`.

Each bearing in XJTU-SY has two sensor channels:
- Assay 1 → horizontal accelerometer
- Assay 2 → vertical accelerometer

In [ ]:
# Define the group once — reuse across any study
# Assay 1 = horizontal accelerometer, Assay 2 = vertical accelerometer
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")

print(both_channels)

## 2. Lifecycle features for both channels

Pass the group directly to `study.lifecycle_features(group=...)` — no intermediate binding object to name.  
Returns `dict[label, DataFrame]` — one entry per assay in the group.


In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")
study = wrapper.study(1)  # first study (1-based index)

lc_both = study.lifecycle_features(both_channels, file_type="raw", n_workers=4)

for label, df in lc_both.items():
    print(f"{label}: {len(df)} runs, columns = {list(df.columns)}")


## 3. Plot the grouped channels side by side

Pass the dict directly to `plot_multi_lifecycle()` to overlay both channels.

In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")
study = wrapper.study(1)  # first study (1-based index)
lc_both = study.lifecycle_features(both_channels, file_type="raw", n_workers=4)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    lc_both,
    feature="rms",
    title="Study 1 — Horizontal vs Vertical RMS Degradation",
)
bokeh_show(fig)

In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")
study = wrapper.study(1)  # first study (1-based index)
lc_both = study.lifecycle_features(both_channels, file_type="raw", n_workers=4)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    lc_both,
    feature="kurtosis",
    title="Study 1 — Horizontal vs Vertical Kurtosis",
)
bokeh_show(fig)

## 4. Cross-study comparison with a group

Call `compare_with()` directly on the **descriptor** (`both_channels`) — no study binding needed.  
All studies are explicit: what you pass is exactly what you get.


In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")

study_1 = wrapper.study(1)  # first study (1-based index)
study_2 = wrapper.study(2)  # second study (1-based index)

# Call compare_with on the descriptor — no prior study.assay_group() needed
cross = both_channels.compare_with(
    [study_1, study_2],
    file_type="raw",
    n_workers=4,
)

print(f"{len(cross)} traces loaded:")
for key in cross:
    print(f"  {key}")


In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")
cross = both_channels.compare_with([wrapper.study(1), wrapper.study(2)], file_type="raw", n_workers=4)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    cross,
    feature="rms",
    title="Both Channels — Studies 1 and 2",
)
bokeh_show(fig)

## 5. Use compare_studies with an AssayGroup

`wrapper.compare_studies()` also accepts an `assay_group` argument — this is the shortest form for large comparisons.

Only the studies you pass are included.

In [ ]:
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")
plotter = ISAPlotter()

lc_all = wrapper.compare_studies(
    [1, 2],  # first and second study (1-based indices)
    assay_group=both_channels,
    file_type="raw",
    n_workers=4,
)

fig = plotter.plot_multi_lifecycle(
    lc_all,
    feature="rms",
    title="compare_studies() with AssayGroup — Studies 1 and 2",
)
bokeh_show(fig)